# Normalize and cache the business records

Select the **ml-challenge-er** kernel and run cells from top to bottom. This notebook processes all six source TSVs, plus a separate ground-truth cache. It does **not** train a model or generate candidate pairs.

- Original columns and entity IDs are preserved. Normalized fields are additional views.
- ICU performs **offline transliteration**, not translation. Romanized names are approximate; addresses still matter.
- No business lookups, downloads, or external data services are used.
- Processing uses CPU batches and compressed Parquet; CUDA is unnecessary here.
- Completed batches are checkpointed. Interrupt safely, then rerun the processing cell to resume.

The pilot below runs first. Its estimate is a planning aid, not a guaranteed finish time. Avoid running two copies against the same cache directory.

Submission location: `code/business_entity_resolution/src/01_normalize_and_cache.ipynb`.
Pinned dependencies and reproduction instructions live one directory above this notebook.
Cache files are intermediates; they are not the two TSVs uploaded for scoring.


In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "8")
import atexit
import ctypes
import gc
import json
import shutil
import sys
import time
from pathlib import Path

import polars as pl
from IPython.display import display

print("Python:", sys.executable)
print("Polars:", pl.__version__, "| threads:", pl.thread_pool_size())
assert pl.__version__ == "1.44.2", "Select the pinned ml-challenge-er environment; see ../environment.yml."
assert sys.platform.startswith("linux"), "This pinned environment and ICU loader target Linux."

# Use ONLY ICU libraries shipped by the selected conda environment, never /usr/bin/uconv.
# This small wrapper calls the documented public ICU C API using Python's standard library.
# https://unicode-org.github.io/icu-docs/apidoc/released/icu4c/utrans_8h.html
ICU_MAJOR = 78
ICU_LIBDIR = Path(sys.prefix) / "lib"
_icu_libs = []
for name in ("icudata", "icuuc", "icui18n"):
    path = ICU_LIBDIR / f"lib{name}.so.{ICU_MAJOR}"
    assert path.exists(), f"Missing {path}. Create the pinned conda environment from ../environment.yml."
    _icu_libs.append(ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL))

def icu_symbol(library, name):
    return getattr(library, f"{name}_{ICU_MAJOR}")

_get_version = icu_symbol(_icu_libs[1], "u_getVersion")
_get_version.argtypes = [ctypes.POINTER(ctypes.c_uint8)]
_get_version.restype = None
_version = (ctypes.c_uint8 * 4)()
_get_version(_version)
ICU_VERSION = ".".join(map(str, list(_version)[:2]))
assert ICU_VERSION == "78.3", f"Expected pinned ICU 78.3, got {ICU_VERSION}."

class OfflineTransliterator:
    """Own an ICU transliterator and grow UTF-16 buffers when needed. No network IO."""
    def __init__(self, rules):
        self.encoding = "utf-16-le" if sys.byteorder == "little" else "utf-16-be"
        uchar = ctypes.POINTER(ctypes.c_uint16)
        i32ptr = ctypes.POINTER(ctypes.c_int32)
        self.open = icu_symbol(_icu_libs[2], "utrans_openU")
        self.open.argtypes = [uchar, ctypes.c_int32, ctypes.c_int32, uchar,
                              ctypes.c_int32, ctypes.c_void_p, i32ptr]
        self.open.restype = ctypes.c_void_p
        self.apply = icu_symbol(_icu_libs[2], "utrans_transUChars")
        self.apply.argtypes = [ctypes.c_void_p, uchar, i32ptr, ctypes.c_int32,
                               ctypes.c_int32, i32ptr, i32ptr]
        self.apply.restype = None
        self.release = icu_symbol(_icu_libs[2], "utrans_close")
        self.release.argtypes = [ctypes.c_void_p]
        self.release.restype = None
        encoded = rules.encode(self.encoding)
        identifier = (ctypes.c_uint16 * (len(encoded)//2)).from_buffer_copy(encoded)
        status = ctypes.c_int32(0)
        self.handle = self.open(identifier, len(identifier), 0, None, 0, None, ctypes.byref(status))
        if status.value > 0 or not self.handle:
            raise RuntimeError(f"ICU open failed: {status.value}")
        atexit.register(self.close)

    def close(self):
        if self.handle:
            self.release(self.handle)
            self.handle = None

    def transliterate(self, text):
        assert self.handle, "Transliterator is closed. Rerun its configuration cell."
        encoded = text.encode(self.encoding)
        units = len(encoded)//2
        capacity = max(32, units * 3 + 64)
        for _ in range(8):
            buffer = (ctypes.c_uint16 * capacity)()
            ctypes.memmove(buffer, encoded, len(encoded))
            length, limit, status = ctypes.c_int32(units), ctypes.c_int32(units), ctypes.c_int32(0)
            self.apply(self.handle, buffer, ctypes.byref(length), capacity, 0,
                       ctypes.byref(limit), ctypes.byref(status))
            if status.value == 15:  # U_BUFFER_OVERFLOW_ERROR: rerun from unmodified input.
                capacity = max(capacity * 2, length.value + 64)
                continue
            if status.value > 0:
                raise RuntimeError(f"ICU transliteration failed: {status.value}")
            assert 0 <= length.value <= capacity
            return ctypes.string_at(buffer, length.value * 2).decode(self.encoding)
        raise RuntimeError("ICU buffer growth exceeded its bound")

print("Offline ICU:", ICU_VERSION, "from", ICU_LIBDIR)


Python: /home/aline/.conda/envs/ml-challenge-er/bin/python
Polars: 1.44.2 | threads: 8
Offline ICU: 78.3 from /home/aline/.conda/envs/ml-challenge-er/lib


In [2]:
# Portable path selection: set ER_DATA_ROOT to a dataset/ directory containing train/ and test/.
# Local checkout and extracted submission layouts are detected automatically.
ancestors = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((p for p in ancestors if (p / "6ab10eb3b23ba_student_resource/student_resource/dataset").is_dir()), Path.cwd())
if os.environ.get("ER_DATA_ROOT"):
    DATA_ROOT = Path(os.environ["ER_DATA_ROOT"]).expanduser().resolve()
else:
    candidates = [p / relative for p in ancestors for relative in
                  ("dataset", "student_resource/dataset", "6ab10eb3b23ba_student_resource/student_resource/dataset")]
    DATA_ROOT = next((p for p in candidates if (p / "train/train_source1.tsv").is_file()), None)
assert DATA_ROOT is not None, "Set ER_DATA_ROOT or DATA_ROOT to the supplied dataset/ directory."
CACHE_ROOT = Path(os.environ.get("ER_CACHE_ROOT", str(PROJECT_ROOT / "cache/normalized_v1"))).expanduser().resolve()
BATCH_ROWS = 100_000
PILOT_ROWS_PER_FILE = 10_000
PIPELINE_VERSION = "1.1.0"  # New CACHE_ROOT required after changing normalization rules.
FILES = [DATA_ROOT / split / f"{split}_source{source}.tsv"
         for split in ("train", "test") for source in (1, 2, 3)]
RAW_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
ICU_RULES = "Any-Latin; Latin-ASCII; Lower"
if "TRANSLITERATOR" in globals():
    TRANSLITERATOR.close()
TRANSLITERATOR = OfflineTransliterator(ICU_RULES)
assert all(p.is_file() for p in FILES)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_ROOT)
print("Cache:", CACHE_ROOT)
print("Free disk: %.1f GiB" % (shutil.disk_usage(CACHE_ROOT).free / 2**30))


Data: /home/aline/Projects/ML-Challenge/6ab10eb3b23ba_student_resource/student_resource/dataset
Cache: /home/aline/Projects/ML-Challenge/cache/normalized_v1
Free disk: 71.0 GiB


## What gets added

`name_norm` / `address_norm`: NFKC Unicode, lowercase, collapsed whitespace; punctuation retained.

`name_latin` / `address_latin`: ICU romanized views, including digit transliteration where supported. `*_match`: letters and numbers separated into tokens, with accents folded by ICU. Indic combining marks remain intact in the original-script views.

`name_core`: an additional version without trailing legal suffixes; the full name remains available. `*_tokens`: unique, sorted tokens. `address_numbers`: structured numeric strings such as `266/84`; `address_digit_tokens`: component numbers such as `266` and `84`. `postal_candidates`: 5–6 digit tokens, **not verified postal codes**.

Missingness and non-Latin-script flags prevent missing information from looking like agreement. Country labels remain open-ended, including France. No learned dictionaries, imputation, forced address correction, or row deduplication are applied.

In [3]:
def scan_tsv(path):
    # Explicit UTF-8 and TSV parsing. All columns remain strings, including identifiers.
    return pl.scan_csv(path, separator="\t", infer_schema=False, encoding="utf8",
                       low_memory=True, rechunk=False)


def base_text(column):
    return (pl.col(column).fill_null("").str.normalize("NFKC").str.to_lowercase()
            .str.replace_all(r"[\s\p{Z}\p{Cc}]+", " ").str.strip_chars())


def match_text(column):
    return (pl.col(column).str.replace_all("&", " and ", literal=True)
            .str.replace_all(r"[^\p{L}\p{M}\p{N}]+", " ")
            .str.replace_all(r"\s+", " ").str.strip_chars())


def romanize_unique(values):
    # Whitespace/control characters were removed before newline framing.
    values = values.filter(values.str.contains(r"[^\x00-\x7F]")).unique().to_list()
    if not values:
        return {}
    output = TRANSLITERATOR.transliterate("\n".join(values) + "\n")
    converted = output.split("\n")
    assert converted[-1] == "", "Unexpected ICU output framing."
    converted.pop()
    assert len(converted) == len(values), "ICU changed record boundaries."
    return dict(zip(values, converted))


def normalize_batch(raw):
    assert raw.columns == RAW_COLUMNS, f"Unexpected columns: {raw.columns}"
    df = raw.with_columns(
        base_text("business_name").alias("name_norm"),
        base_text("business_address").alias("address_norm"),
        base_text("country").alias("country_key"),
        pl.col("entity_id").str.slice(0, 2).alias("source"),
    )
    mapping = romanize_unique(pl.concat([df["name_norm"], df["address_norm"]]))
    df = df.with_columns(*[
        (pl.col(f"{field}_norm").replace_strict(mapping, default=pl.col(f"{field}_norm"))
         if mapping else pl.col(f"{field}_norm")).alias(f"{field}_latin")
        for field in ("name", "address")
    ])
    df = df.with_columns(*[
        match_text(f"{field}_latin").alias(f"{field}_match") for field in ("name", "address")
    ], *[
        (pl.col(f"{field}_norm") == "").alias(f"{field}_missing") for field in ("name", "address")
    ], *[
        pl.col(f"{field}_norm").str.contains(r"[\p{L}&&[^\p{Latin}]]").alias(f"{field}_has_nonlatin")
        for field in ("name", "address")
    ])
    suffixes = r"(?:\s+(?:private|pvt|limited|ltd|incorporated|inc|corporation|corp|llc|llp|plc|sarl|sas|eurl|sa))+$"
    df = df.with_columns(
        pl.col("name_match").str.replace(suffixes, "").alias("name_core"),
        *[pl.col(f"{field}_match").str.extract_all(r"[\p{L}\p{M}\p{N}]+")
          .list.unique().list.sort().alias(f"{field}_tokens") for field in ("name", "address")],
        pl.col("address_latin").str.extract_all(r"[0-9]+(?:[-/][0-9]+)*")
          .list.unique().list.sort().alias("address_numbers"),
        pl.col("address_latin").str.extract_all(r"[0-9]+")
          .list.unique().list.sort().alias("address_digit_tokens"),
        pl.col("address_latin").str.extract_all(r"\b[0-9]{5,6}\b")
          .list.unique().list.sort().alias("postal_candidates"),
    )
    assert df.height == raw.height
    return df


In [4]:
# Quick invariants: preserve records, missing values, French accents and address numbers.
fixture = pl.DataFrame({
    "entity_id": ["S1-001", "S2-002", "S3-003", "S2-004"],
    "business_name": ["Ss Estate", "एसएस एस्टेट", "ÉCOLE & FILS SAS", "A Private Limited"],
    "business_address": ["House No.266/84", "HOUSE NO.२६६/८४", None, "12 Rue de Paris, 75001"],
    "country": ["India", "India", "France", "France"],
})
checked = normalize_batch(fixture)
assert checked.select(RAW_COLUMNS).equals(fixture)
assert checked["address_missing"].to_list() == [False, False, True, False]
assert checked["address_numbers"][1].to_list() == ["266/84"]
assert checked["name_match"][2] == "ecole and fils sas"
assert checked["name_core"][3] == "a"
assert checked["address_tokens"][2].to_list() == []
assert checked["country_key"].to_list() == ["india", "india", "france", "france"]
display(checked.select("business_name", "name_match", "name_core", "address_numbers", "address_missing"))
print("Normalization checks passed. Transliteration spellings may differ from English names.")

assert TRANSLITERATOR.transliterate("🧪 École\n१२३\n") == "🧪 ecole\n123\n"
assert TRANSLITERATOR.transliterate("ﷺ")  # Exercises expansion beyond initial UTF-16 allocation.


business_name,name_match,name_core,address_numbers,address_missing
str,str,str,list[str],bool
"""Ss Estate""","""ss estate""","""ss estate""","[""266/84""]",false
"""एसएस एस्टेट""","""esa esa esteta""","""esa esa esteta""","[""266/84""]",false
"""ÉCOLE & FILS SAS""","""ecole and fils sas""","""ecole and fils""",[],true
"""A Private Limited""","""a private limited""","""a""","[""12"", ""75001""]",false


Normalization checks passed. Transliteration spellings may differ from English names.


## Pilot: check the output and estimate the full run

This reads the first 10,000 rows of every file, normalizes them and times Parquet writing. The sample is not guaranteed representative. The estimate includes linear extrapolation with a 2× planning margin; full-run progress provides an updated ETA. Temporary pilot Parquet is removed automatically.

In [5]:
import tempfile
pilot_results = []
for path in FILES:
    total_rows = scan_tsv(path).select(pl.len()).collect(engine="streaming").item()
    started = time.perf_counter()
    raw = scan_tsv(path).head(PILOT_ROWS_PER_FILE).collect(engine="streaming")
    normalized = normalize_batch(raw)
    with tempfile.TemporaryDirectory(dir=CACHE_ROOT, prefix="pilot-") as temp:
        target = Path(temp) / "pilot.parquet"
        normalized.write_parquet(target, compression="zstd", compression_level=3)
        size = target.stat().st_size
    seconds = time.perf_counter() - started
    pilot_results.append({"file": path.name, "rows": total_rows, "pilot_rows": raw.height,
                          "pilot_seconds": round(seconds, 3),
                          "estimated_minutes": round(seconds * total_rows / raw.height / 60, 2),
                          "estimated_GiB": round(size * total_rows / raw.height / 2**30, 3)})
    print(path.name, "pilot finished", flush=True)
display(pl.DataFrame(pilot_results))
minutes = sum(r["estimated_minutes"] for r in pilot_results)
gib = sum(r["estimated_GiB"] for r in pilot_results)
print(f"Pilot extrapolation: {minutes:.1f} minutes; plan for roughly {minutes:.1f}–{2*minutes:.1f} minutes or more.")
print(f"Estimated source cache: {gib:.1f} GiB; free disk: {shutil.disk_usage(CACHE_ROOT).free / 2**30:.1f} GiB.")
print("Estimates exclude ground-truth caching; source text stays unchanged.")
display(normalized.select("entity_id", "business_name", "name_match", "business_address", "address_match").head(8))
del raw, normalized
gc.collect()


/tmp/ipykernel_36610/2832864975.py:3: DeprecationWarning: `rechunk` parameter on scan_csv() will be removed. Consider first collecting the scan to a DataFrame, then calling df.rechunk() on the result.
(Deprecated in version 1.43.2)
  return pl.scan_csv(path, separator="\t", infer_schema=False, encoding="utf8",


train_source1.tsv pilot finished
train_source2.tsv pilot finished
train_source3.tsv pilot finished
test_source1.tsv pilot finished
test_source2.tsv pilot finished
test_source3.tsv pilot finished


file,rows,pilot_rows,pilot_seconds,estimated_minutes,estimated_GiB
str,i64,i64,f64,f64,f64
"""train_source1.tsv""",2206821,10000,0.048,0.18,0.384
"""train_source2.tsv""",5034616,10000,0.159,1.33,0.909
"""train_source3.tsv""",5285603,10000,0.117,1.03,0.945
"""test_source1.tsv""",1732544,10000,0.063,0.18,0.307
"""test_source2.tsv""",4887273,10000,0.212,1.73,0.915
"""test_source3.tsv""",5082316,10000,0.148,1.26,0.926


Pilot extrapolation: 5.7 minutes; plan for roughly 5.7–11.4 minutes or more.
Estimated source cache: 4.4 GiB; free disk: 71.0 GiB.
Estimates exclude ground-truth caching; source text stays unchanged.


entity_id,business_name,name_match,business_address,address_match
str,str,str,str,str
"""S3-462677478""","""मॉडर्न फाइनेंस""","""modarna pha inensa""","""No 10 Enkay Square, 448A, Udyo…","""no 10 enkay square 448a udyog …"
"""S3-374810425""","""Shri Sai Infratech Co""","""shri sai infratech co""","""3/115, East Delhi, DL""","""3 115 east delhi dl"""
"""S3-198586129""","""Fractales Amis Groupe S.A.S""","""fractales amis groupe s a s""","""23 Rue Icmre, La Teste-de-buch…","""23 rue icmre la teste de buch …"
"""S3-10300249""","""Shri Supreme Consulting Privat…","""shri supreme consulting privat…","""H.no 910 A 3503, Mumbai, महारा…","""h no 910 a 3503 mumbai maharas…"
"""S3-604980231""","""Prime Realty Ventures Public L…","""prime realty ventures public l…","""G.t. Karnal Road, Industrial A…","""g t karnal road industrial are…"
"""S3-577972146""","""Perfect Investments Private""","""perfect investments private""","""Plot No.99, Flat No.201, Sri D…","""plot no 99 flat no 201 sri dha…"
"""S3-150632472""","""Pace Mfa PC""","""pace mfa pc""","""2132 Thomas Run Road, Bel Air,…","""2132 thomas run road bel air m…"
"""S3-184785497""","""ஈஸ்டர்ன் கன்சல்டன்சி பிரைவேட் …","""istarn kancaltanci piraivet li…","""1410 Sri Mahalakshmi Mandiar 5…","""1410 sri mahalakshmi mandiar 5…"


35

In [6]:
def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def input_signature(path, kind):
    stat = path.stat()
    return {"path": str(path.resolve()), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns,
            "pipeline": PIPELINE_VERSION, "kind": kind, "batch_rows": BATCH_ROWS,
            "polars": pl.__version__, "icu": ICU_VERSION, "icu_rules": ICU_RULES}


def cache_file(path, kind="source"):
    folder = CACHE_ROOT / path.parent.name / path.stem
    folder.mkdir(parents=True, exist_ok=True)
    checkpoint = folder / "manifest.json"
    signature = input_signature(path, kind)
    total_rows = scan_tsv(path).select(pl.len()).collect(engine="streaming").item()
    if checkpoint.exists():
        manifest = json.loads(checkpoint.read_text())
        assert manifest["signature"] == signature, (
            f"Cache configuration or input changed: {folder}. Use a new CACHE_ROOT; do not mix caches.")
    else:
        manifest = {"signature": signature, "total_rows": total_rows, "parts": {}, "complete": False}
        atomic_json(checkpoint, manifest)
    # Only a part with both a valid checkpoint entry and a readable Parquet is reusable.
    verified_parts = set()
    for key, info in manifest["parts"].items():
        part = folder / key
        try:
            if part.stat().st_size == info["bytes"] and pl.scan_parquet(part).select(pl.len()).collect().item() == info["rows"]:
                verified_parts.add(key)
        except (OSError, pl.exceptions.PolarsError):
            pass
    if manifest["complete"] and len(verified_parts) == len(manifest["parts"]) and sum(
        p["rows"] for p in manifest["parts"].values()) == total_rows:
        print(path.name, "already complete — skipped", flush=True)
        return manifest
    manifest["complete"] = False
    atomic_json(checkpoint, manifest)
    started = time.perf_counter()
    processed = newly_written = 0
    chunks_seen = []
    next_report = 0.0
    for index, raw in enumerate(scan_tsv(path).collect_batches(chunk_size=BATCH_ROWS, maintain_order=True, engine="streaming")):
        key = f"part-{index:05d}.parquet"
        chunks_seen.append(key)
        first_id, last_id = raw.item(0, 0), raw.item(raw.height - 1, 0)
        if key in verified_parts:
            info = manifest["parts"][key]
            assert (info["first_id"], info["last_id"], info["rows"]) == (first_id, last_id, raw.height), "Batch boundaries changed; use a new cache directory."
            processed += raw.height
            continue
        if kind == "source":
            assert raw["entity_id"].null_count() == 0
            expected = "S" + path.stem[-1] + "-"
            assert raw["entity_id"].str.starts_with(expected).all(), "Unexpected source IDs."
            result = normalize_batch(raw)
        else:
            assert raw.columns == ["source1_entity_id", "matched_entity_ids"]
            result = raw.with_columns(
                pl.col("matched_entity_ids").fill_null("").str.extract_all(r"S[23]-[^,\s]+")
                  .alias("matched_ids")
            ).with_columns(pl.col("matched_ids").list.len().alias("match_count"))
        target = folder / key
        temporary = folder / (key + ".tmp")
        result.write_parquet(temporary, compression="zstd", compression_level=3, row_group_size=25_000)
        temporary.replace(target)
        manifest["parts"][key] = {"rows": raw.height, "bytes": target.stat().st_size,
                                    "first_id": first_id, "last_id": last_id}
        atomic_json(checkpoint, manifest)
        processed += raw.height
        newly_written += raw.height
        elapsed = time.perf_counter() - started
        if elapsed >= next_report or processed == total_rows:
            rate = newly_written / max(elapsed, 0.001)
            eta = (total_rows - processed) / max(rate, 0.001) / 60
            print(f"{path.name}: {processed:,}/{total_rows:,} ({100*processed/total_rows:.1f}%) | {rate:,.0f} new rows/s | ETA ~{eta:.1f} min", flush=True)
            next_report = elapsed + 15
        del result
    assert processed == total_rows
    assert set(chunks_seen) == set(manifest["parts"]), "Unexpected stale cache parts."
    assert sum(p["rows"] for p in manifest["parts"].values()) == total_rows
    assert input_signature(path, kind) == signature, "Input file changed during processing."
    manifest["complete"] = True
    manifest["last_run_seconds"] = round(time.perf_counter() - started, 3)
    atomic_json(checkpoint, manifest)
    print(path.name, "complete:", f"{total_rows:,}", "rows", flush=True)
    return manifest


## Run the full cache

This is the full processing cell. It starts immediately when executed; **Run All includes this step**. Keep the machine awake. To pause, interrupt the kernel; rerunning this cell reuses committed batches. Do not change batch size, normalization rules, or dependency versions while resuming an existing cache.

Only manifests marked `complete: true` should be used by later matching notebooks. The label cache is separate and must never be used to normalize test records or retrieve validation candidates.

In [7]:
started = time.perf_counter()
for path in FILES:
    cache_file(path)
    gc.collect()
cache_file(DATA_ROOT / "train/train_ground_truth.tsv", kind="ground_truth")
print(f"All caches finished in {(time.perf_counter()-started)/60:.1f} minutes this run.")


/tmp/ipykernel_36610/2832864975.py:3: DeprecationWarning: `rechunk` parameter on scan_csv() will be removed. Consider first collecting the scan to a DataFrame, then calling df.rechunk() on the result.
(Deprecated in version 1.43.2)
  return pl.scan_csv(path, separator="\t", infer_schema=False, encoding="utf8",


train_source1.tsv: 100,000/2,206,821 (4.5%) | 409,983 new rows/s | ETA ~0.1 min
train_source1.tsv: 2,206,821/2,206,821 (100.0%) | 449,252 new rows/s | ETA ~0.0 min
train_source1.tsv complete: 2,206,821 rows
train_source2.tsv: 100,000/5,034,616 (2.0%) | 13,577 new rows/s | ETA ~6.1 min
train_source2.tsv: 400,000/5,034,616 (7.9%) | 13,927 new rows/s | ETA ~5.5 min
train_source2.tsv: 700,000/5,034,616 (13.9%) | 13,857 new rows/s | ETA ~5.2 min
train_source2.tsv: 1,000,000/5,034,616 (19.9%) | 13,916 new rows/s | ETA ~4.8 min
train_source2.tsv: 1,300,000/5,034,616 (25.8%) | 13,802 new rows/s | ETA ~4.5 min
train_source2.tsv: 1,600,000/5,034,616 (31.8%) | 13,872 new rows/s | ETA ~4.1 min
train_source2.tsv: 1,900,000/5,034,616 (37.7%) | 13,844 new rows/s | ETA ~3.8 min
train_source2.tsv: 2,200,000/5,034,616 (43.7%) | 13,809 new rows/s | ETA ~3.4 min
train_source2.tsv: 2,500,000/5,034,616 (49.7%) | 13,769 new rows/s | ETA ~3.1 min
train_source2.tsv: 2,800,000/5,034,616 (55.6%) | 13,748 new row

In [8]:
# Verify the completed outputs and show where the next notebook will read them.
summary = []
for path in FILES + [DATA_ROOT / "train/train_ground_truth.tsv"]:
    folder = CACHE_ROOT / path.parent.name / path.stem
    manifest = json.loads((folder / "manifest.json").read_text())
    assert manifest["complete"], f"Incomplete cache: {folder}"
    assert manifest["signature"] == input_signature(path, "source" if path in FILES else "ground_truth")
    parts = [folder / key for key in sorted(manifest["parts"])]
    cached = pl.scan_parquet(parts)
    actual = cached.select(pl.len()).collect().item()
    assert actual == manifest["total_rows"]
    summary.append({"file": path.name, "rows": actual, "parts": len(parts),
                    "cache_GiB": round(sum(p.stat().st_size for p in parts) / 2**30, 3)})
display(pl.DataFrame(summary))
print("Verified: all source and label caches are complete, with matching row counts.")


file,rows,parts,cache_GiB
str,i64,i64,f64
"""train_source1.tsv""",2206821,23,0.377
"""train_source2.tsv""",5034616,51,0.904
"""train_source3.tsv""",5285603,53,0.938
"""test_source1.tsv""",1732544,18,0.307
"""test_source2.tsv""",4887273,49,0.908
"""test_source3.tsv""",5082316,51,0.92
"""train_ground_truth.tsv""",2206821,23,0.099


Verified: all source and label caches are complete, with matching row counts.


## Read the cache without loading everything into RAM

Keep downstream processing lazy or batched. An unrestricted `.collect()` across all six sources would consume substantial memory. Use only the columns needed for each matching stage.

In [9]:
def open_cache(split, source):
    folder = CACHE_ROOT / split / f"{split}_source{source}"
    manifest = json.loads((folder / "manifest.json").read_text())
    assert manifest["complete"], "Finish normalization before using this cache."
    return pl.scan_parquet([folder / key for key in sorted(manifest["parts"])])

train_reference = open_cache("train", 1)
display(train_reference.select("entity_id", "country", "name_match", "address_match", "address_numbers").head(10).collect())
# Example for later: open_cache("test", 2).filter(pl.col("country_key") == "france")


entity_id,country,name_match,address_match,address_numbers
str,str,str,str,list[str]
"""S1-925783039""","""US""","""orelee s barbershop""","""1795 westchester drive high po…","[""1795""]"
"""S1-773889195""","""US""","""prime money""","""17560 ellis road tahlequah ok""","[""17560""]"
"""S1-377745466""","""US""","""b retail inc""","""1712 montebello avenue phoenix…","[""1712""]"
"""S1-133037285""","""US""","""christ chapel""","""2100 cameron drive unit apartm…","[""2100""]"
"""S1-755362802""","""India""","""prabhav business center""","""797 lake town block a kolkata …","[""797""]"
"""S1-851869949""","""US""","""custom wealth services llc""","""oh columbus 5559 orville avenu…","[""5559""]"
"""S1-785847572""","""India""","""consulting nyasa nursing priva…","""2505 tower 1 oakwood runwal gr…","[""1"", ""2505""]"
"""S1-27541239""","""US""","""nexus anchor rain""","""1111 church street unit 2007 n…","[""1111"", ""2007""]"
"""S1-629417405""","""US""","""moore bitwise inc""","""337 oakland avenue michigan ci…","[""337""]"
